In [1]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine


# database connections 

In [2]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)
    

# Extract

In [3]:
dim_ips = pd.read_sql_table('ips', co_sa)
dim_ips.info()


<class 'pandas.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id_ips        86 non-null     str  
 1   tipo_ips      86 non-null     str  
 2   nombre        86 non-null     str  
 3   direccion     86 non-null     str  
 4   nivel         86 non-null     str  
 5   municipio     86 non-null     str  
 6   departamento  86 non-null     str  
dtypes: str(7)
memory usage: 4.8 KB


# Transformations

In [4]:
dim_ips.replace({'':'0'},inplace=True)

,id_ips,tipo_ips,nombre,direccion,nivel,municipio,departamento
0,IPS_1,Clinica,Clinica de Occidente,Kra 76 # 38-102,0,Cali,Valle del Cauca
1,IPS_2,Clinica,Clinica Valle del Lili,Diagonal 98 # 50-59,0,Cali,Valle del Cauca
2,IPS_3,Hospital,Hospital Infantil Clínica Noel,Kra 72 # 85-17,3,Cali,Valle del Cauca
3,IPS_4,Hospital,Hospital Universitario del Valle Evaristo García,Transversal 94 # 75-74,4,Cali,Valle del Cauca
4,IPS_5,Hospital,Hospital San Juan de Dios,Diagonal 105 # 66-119,1,Cali,Valle del Cauca
...,...,...,...,...,...,...,...
81,IPS_82,Consultorio,Consultorio Leonor Carrasco Moyano,Kra 8 # 87-32,0,Villavicencio,Meta
82,IPS_83,Consultorio,Consultorio Cecilia Avila Arce,Avenida 154 # 13-108,0,Villavicencio,Meta
83,IPS_84,Consultorio,Consultorio Calderón,Avenida 7 # 54-24,0,Girardot,Cundinamarca
84,IPS_85,Consultorio,Consultorio Serrano Crespo,Kra 18 # 28-64,0,Valparaíso,Caqueta


In [5]:
dim_ips["saved"] = date.today()
dim_ips.describe(include='all')

,id_ips,tipo_ips,nombre,direccion,nivel,municipio,departamento,saved
count,86,86,86,86,86,86,86,86
unique,86,6,83,86,6,23,7,1
top,IPS_1,Hospital,Hospital Isaías Duarte Cancio,Kra 76 # 38-102,0,Cali,Cundinamarca,2026-05-13
freq,1,27,2,1,59,11,15,86


# load

In [6]:
dim_ips.to_sql('dim_ips', etl_conn, if_exists='replace',index_label='key_dim_ips')

86